# FIFA World Cup 2026 — Player Stats Analysis

Starter notebook for the dataset produced by `scrape_wc26.py`.

- **Load** the combined master parquet (one row per player per game).
- **Per-90 normalization** so players with different minutes are comparable.
- **Player & team leaderboards**.

Re-run `python3 scrape_wc26.py` to add newly-played games, then just re-run this notebook.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

DATA = Path("data")
df = pd.read_parquet(DATA / "all_players_stats.parquet")
glossary = pd.read_csv(DATA / "glossary.csv")

# Identity (non-stat) columns; everything else is a numeric stat.
ID_COLS = [
    "game_id", "match_date", "competition", "season",
    "team_id", "team_name", "home_away", "is_winner", "formation",
    "team_score", "opp_team_id", "opp_team_name", "opp_score",
    "player_id", "player_name", "jersey", "position", "position_abbr",
    "starter", "subbed_in", "subbed_out", "minutes", "appearances",
    "scraped_at",
]
STAT_COLS = [c for c in df.columns if c not in ID_COLS]
df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")

print(f"{len(df)} player-rows | {df.game_id.nunique()} games | "
      f"{df.player_id.nunique()} players | {len(STAT_COLS)} stat fields")
df.head(3)

376 player-rows | 12 games | 376 players | 143 stat fields


,game_id,match_date,competition,season,team_id,team_name,home_away,is_winner,formation,team_score,opp_team_id,opp_team_name,opp_score,player_id,player_name,jersey,position,position_abbr,starter,subbed_in,subbed_out,minutes,appearances,scraped_at,accurateCrosses,accurateKeeperSweeper,accurateLongBalls,accuratePasses,accurateThroughBalls,aerialDuelPct,...,starts,subIns,subOuts,successfulFinalThirdPasses,suspensions,tacklePct,throughBallPct,timeEnded,timeStarted,timesTackled,totalClearance,totalContest,totalCrosses,totalFinalThirdPasses,totalGoals,totalKeeperSweeper,totalLongBalls,totalPasses,totalShots,totalTackles,totalThroughBalls,touches,touchesInOppBox,turnover,unclaimedCrosses,winPct,wins,wonContest,wonCorners,yellowCards
0,760414,2026-06-12 02:00:00+00:00,FIFA World Cup,2026,451,South Korea,home,True,3-4-2-1,2,450,Czechia,1,175921,Kim Seung-Gyu,1,Goalkeeper,G,True,False,False,90.0,1.0,2026-06-12T23:24:11+00:00,0.0,1.0,8.0,19.0,0.0,NaN,...,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,12.0,0.0,1.0,23.0,34.0,0.0,0.0,0.0,39.0,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0.0
1,760414,2026-06-12 02:00:00+00:00,FIFA World Cup,2026,451,South Korea,home,True,3-4-2-1,2,450,Czechia,1,157688,Kim Min-Jae,4,Center Defender,CD,True,False,False,90.0,1.0,2026-06-12T23:24:12+00:00,0.0,NaN,1.0,51.0,0.0,0.800,...,1.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,0.0,4.0,0.0,NaN,3.0,54.0,0.0,1.0,0.0,64.0,NaN,NaN,0.0,0.0,0.0,NaN,0.0,0.0
2,760414,2026-06-12 02:00:00+00:00,FIFA World Cup,2026,451,South Korea,home,True,3-4-2-1,2,450,Czechia,1,345769,Lee Gi-Hyuk,3,Center Left Defender,CD-L,True,False,False,90.0,1.0,2026-06-12T23:24:12+00:00,0.0,NaN,4.0,58.0,0.0,0.667,...,1.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,NaN,0.0,9.0,0.0,NaN,6.0,62.0,0.0,0.0,0.0,76.0,NaN,1.0,0.0,0.0,0.0,NaN,0.0,1.0


## Glossary lookup

Every stat column mapped to its human name, category and on-page abbreviation.

In [2]:
def describe(*cols):
    """Show glossary rows for the given stat column name(s)."""
    return glossary[glossary.column.isin(cols)][
        ["column", "name", "abbreviation", "category", "description"]
    ]

# Example: what are these columns?
describe("expectedGoals", "expectedAssists", "touches",
         "defensiveInterventions", "duelsWon", "progressiveCarries")

,column,name,abbreviation,category,description
2,defensiveInterventions,defensiveInterventions,DINT,defensive,"The sum of ball recoveries, won tackles, effec..."
28,duelsWon,duelsWon,DUELW,general,Total number of duels for possession of the ba...
48,touches,touches,TCH,general,Total sum of a player's or team's on-the-ball ...
92,expectedAssists,expectedAssists,xA,offensive,Total Expected Assists
95,expectedGoals,expectedGoals,xG,offensive,Total expected goals
123,progressiveCarries,progressiveCarries,PRGC,offensive,Carries that significantly move the ball towar...


## Per-90 normalization

Most volume stats scale with minutes played. To compare a sub who played 20'
against a starter who played 90', we express counting stats **per 90 minutes**.

Rate/percentage stats (already normalized) and a few non-additive fields are
left as-is.

In [3]:
# Stats that are already rates/percentages or otherwise shouldn't be per-90'd.
NON_PER90_SUBSTR = ("Pct", "pct", "Rating", "win", "Win")
NON_PER90_EXACT = {
    "appearances", "starts", "dnp", "wins", "losses", "draws",
    "cleanSheet", "partialCleenSheet", "timeStarted", "timeEnded",
    "goalDifference", "suspensions",
}

def is_rate_col(c):
    return c in NON_PER90_EXACT or any(s in c for s in NON_PER90_SUBSTR)

PER90_COLS = [c for c in STAT_COLS if not is_rate_col(c)]

def add_per90(frame, min_minutes=0):
    """Return a copy with `<stat>_p90` columns for all counting stats."""
    out = frame.copy()
    mins = out["minutes"].replace(0, np.nan)
    for c in PER90_COLS:
        out[f"{c}_p90"] = out[c] / mins * 90
    if min_minutes:
        out = out[out["minutes"] >= min_minutes]
    return out

df90 = add_per90(df)
df90[["player_name", "team_name", "minutes",
      "expectedGoals", "expectedGoals_p90",
      "touches", "touches_p90"]].head()

/var/folders/c3/fp20mf856mn7bnz2vyhl2jp40000gp/T/ipykernel_35455/3105648083.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_p90"] = out[c] / mins * 90
/var/folders/c3/fp20mf856mn7bnz2vyhl2jp40000gp/T/ipykernel_35455/3105648083.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_p90"] = out[c] / mins * 90
/var/folders/c3/fp20mf856mn7bnz2vyhl2jp40000gp/T/ipykernel_35455/3105648083.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

,player_name,team_name,minutes,expectedGoals,expectedGoals_p90,touches,touches_p90
0,Kim Seung-Gyu,South Korea,90.0,NaN,NaN,39.0,39.000000
1,Kim Min-Jae,South Korea,90.0,NaN,NaN,64.0,64.000000
2,Lee Gi-Hyuk,South Korea,90.0,NaN,NaN,76.0,76.000000
3,Lee Han-Beom,South Korea,90.0,0.043,0.043,80.0,80.000000
4,Paik Seung-Ho,South Korea,84.0,NaN,NaN,72.0,77.142857


## Player leaderboards

Two flavours:
- **Tournament totals** — sum a stat across all games a player has featured in.
- **Per-90 (rate)** — average output per 90', with a minutes filter so tiny
  samples don't top the chart.

In [4]:
def player_totals(stat, n=15):
    """Top-n players by cumulative total of `stat` across all games."""
    g = (df.groupby(["player_id", "player_name", "team_name"], as_index=False)
           .agg(games=("game_id", "nunique"),
                minutes=("minutes", "sum"),
                total=(stat, "sum")))
    return g.sort_values("total", ascending=False).head(n).reset_index(drop=True)

def player_per90(stat, n=15, min_minutes=90):
    """Top-n players by `stat` per 90', requiring >= min_minutes total."""
    g = (df.groupby(["player_id", "player_name", "team_name"], as_index=False)
           .agg(games=("game_id", "nunique"),
                minutes=("minutes", "sum"),
                total=(stat, "sum")))
    g = g[g.minutes >= min_minutes].copy()
    g[f"{stat}_p90"] = g["total"] / g["minutes"] * 90
    return g.sort_values(f"{stat}_p90", ascending=False).head(n).reset_index(drop=True)

print("Top expected-goals (xG) — tournament totals")
player_totals("expectedGoals")

Top expected-goals (xG) — tournament totals


,player_id,player_name,team_name,games,minutes,total
0,202099,Breel Embolo,Switzerland,1,90.0,1.144
1,149945,Son Heung-Min,South Korea,1,69.0,1.014
2,231182,Kai Havertz,Germany,1,90.0,1.012
3,167060,Raúl Jiménez,Mexico,1,76.0,0.844
4,202641,Leroy Sané,Germany,1,90.0,0.821
5,304572,Ismael Saibari,Morocco,1,89.0,0.707
6,305118,Jovo Lukic,Bosnia-Herzegovina,1,62.0,0.702
7,301894,Igor Thiago,Brazil,1,62.0,0.665
8,250241,Frantzdy Pierrot,Haiti,1,90.0,0.649
9,301416,Malik Tillman,United States,1,82.0,0.646


In [5]:
print("Top xG per 90' (min 90 minutes played)")
player_per90("expectedGoals", min_minutes=90)

Top xG per 90' (min 90 minutes played)


,player_id,player_name,team_name,games,minutes,total,expectedGoals_p90
0,202099,Breel Embolo,Switzerland,1,90.0,1.144,1.144
1,231182,Kai Havertz,Germany,1,90.0,1.012,1.012
2,202641,Leroy Sané,Germany,1,90.0,0.821,0.821
3,250241,Frantzdy Pierrot,Haiti,1,90.0,0.649,0.649
4,258906,Viktor Gyökeres,Sweden,1,90.0,0.633,0.633
5,332081,Alessandro Circati,Australia,1,90.0,0.507,0.507
6,359370,Aleksandar Pavlovic,Germany,1,90.0,0.398,0.398
7,212237,Nico Elvedi,Switzerland,1,90.0,0.311,0.311
8,286828,Nico Schlotterbeck,Germany,1,90.0,0.309,0.309
9,230666,Tomás Soucek,Czechia,1,90.0,0.279,0.279


In [6]:
# Swap in any stat column. A few ideas:
print("Most touches per 90'")
display(player_per90("touches", min_minutes=90, n=10))

print("\nMost progressive carries per 90'")
display(player_per90("progressiveCarries", min_minutes=90, n=10))

print("\nMost defensive interventions per 90'")
display(player_per90("defensiveInterventions", min_minutes=90, n=10))

Most touches per 90'


,player_id,player_name,team_name,games,minutes,total,touches_p90
0,238318,Ferdi Kadioglu,Türkiye,1,90.0,124.0,124.0
1,170785,Hakan Çalhanoglu,Türkiye,1,90.0,120.0,120.0
2,193149,Abdülkerim Bardakçi,Türkiye,1,90.0,115.0,115.0
3,157892,Virgil van Dijk,Netherlands,1,90.0,114.0,114.0
4,214562,Manuel Akanji,Switzerland,1,90.0,109.0,109.0
5,282583,Jan Paul van Hecke,Netherlands,1,90.0,106.0,106.0
6,310452,Arda Güler,Türkiye,1,90.0,105.0,105.0
7,212237,Nico Elvedi,Switzerland,1,90.0,104.0,104.0
8,286828,Nico Schlotterbeck,Germany,1,90.0,104.0,104.0
9,146290,Tim Ream,United States,1,90.0,102.0,102.0



Most progressive carries per 90'


,player_id,player_name,team_name,games,minutes,total,progressiveCarries_p90
0,212237,Nico Elvedi,Switzerland,1,90.0,22.0,22.0
1,282583,Jan Paul van Hecke,Netherlands,1,90.0,20.0,20.0
2,214562,Manuel Akanji,Switzerland,1,90.0,20.0,20.0
3,238318,Ferdi Kadioglu,Türkiye,1,90.0,19.0,19.0
4,146290,Tim Ream,United States,1,90.0,16.0,16.0
5,296982,Mohamed Amine Ben Hamida,Tunisia,1,90.0,16.0,16.0
6,402479,Yan Diomande,Ivory Coast,1,90.0,15.0,15.0
7,193149,Abdülkerim Bardakçi,Türkiye,1,90.0,15.0,15.0
8,170785,Hakan Çalhanoglu,Türkiye,1,90.0,15.0,15.0
9,259975,Johan Vásquez,Mexico,1,90.0,15.0,15.0



Most defensive interventions per 90'


,player_id,player_name,team_name,games,minutes,total,defensiveInterventions_p90
0,318390,Tarik Muharemovic,Bosnia-Herzegovina,1,90.0,28.0,28.0
1,220538,Nikola Katic,Bosnia-Herzegovina,1,90.0,25.0,25.0
2,308449,Emmanuel Agbadou,Ivory Coast,1,90.0,17.0,17.0
3,286569,Wilfried Singo,Ivory Coast,1,90.0,16.0,16.0
4,245918,Armando Obispo,Curaçao,1,90.0,15.0,15.0
5,207461,Harry Souttar,Australia,1,90.0,15.0,15.0
6,252521,Hannes Delcroix,Haiti,1,90.0,14.0,14.0
7,332081,Alessandro Circati,Australia,1,90.0,14.0,14.0
8,286828,Nico Schlotterbeck,Germany,1,90.0,14.0,14.0
9,296982,Mohamed Amine Ben Hamida,Tunisia,1,90.0,14.0,14.0


## Multi-stat player profile

A compact attacking + possession + defensive snapshot, per 90'.

In [7]:
PROFILE = [
    "totalGoals", "expectedGoals", "goalAssists", "expectedAssists",
    "totalShots", "shotsOnTarget", "bigChanceCreated",
    "touches", "touchesInOppBox", "progressiveCarries", "passPct",
    "duelsWon", "defensiveInterventions", "interceptions", "totalClearance",
]

def profile_per90(min_minutes=90):
    agg = {c: (c, "sum") for c in PROFILE}
    g = (df.groupby(["player_name", "team_name", "position"], as_index=False)
           .agg(minutes=("minutes", "sum"), **agg))
    g = g[g.minutes >= min_minutes].copy()
    for c in PROFILE:
        if "Pct" not in c:
            g[c] = g[c] / g["minutes"] * 90
    return g.round(2)

prof = profile_per90()
prof.sort_values("expectedGoals", ascending=False).head(12)

,player_name,team_name,position,minutes,totalGoals,expectedGoals,goalAssists,expectedAssists,totalShots,shotsOnTarget,bigChanceCreated,touches,touchesInOppBox,progressiveCarries,passPct,duelsWon,defensiveInterventions,interceptions,totalClearance
57,Breel Embolo,Switzerland,Forward,90.0,1.0,1.14,0.0,0.84,4.0,1.0,2.0,21.0,10.0,1.0,0.75,2.0,0.0,0.0,0.0
188,Kai Havertz,Germany,Forward,90.0,2.0,1.01,0.0,0.26,4.0,2.0,0.0,52.0,11.0,2.0,0.93,2.0,2.0,1.0,1.0
214,Leroy Sané,Germany,Attacking Midfielder Right,90.0,0.0,0.82,0.0,0.13,3.0,0.0,0.0,71.0,10.0,5.0,0.80,4.0,1.0,0.0,0.0
115,Frantzdy Pierrot,Haiti,Center Right Forward,90.0,0.0,0.65,0.0,0.10,3.0,1.0,0.0,16.0,5.0,1.0,0.83,6.0,2.0,0.0,2.0
344,Viktor Gyökeres,Sweden,Center Right Forward,90.0,1.0,0.63,1.0,0.12,5.0,2.0,1.0,38.0,11.0,8.0,0.85,3.0,1.0,0.0,0.0
12,Alessandro Circati,Australia,Center Right Defender,90.0,0.0,0.51,0.0,0.01,1.0,0.0,0.0,52.0,4.0,0.0,0.81,4.0,14.0,1.0,6.0
11,Aleksandar Pavlovic,Germany,Left Midfielder,90.0,0.0,0.40,0.0,0.34,3.0,1.0,0.0,84.0,1.0,5.0,0.93,5.0,7.0,0.0,0.0
259,Nico Elvedi,Switzerland,Center Right Defender,90.0,0.0,0.31,0.0,0.06,2.0,0.0,0.0,104.0,5.0,22.0,0.97,2.0,8.0,2.0,4.0
260,Nico Schlotterbeck,Germany,Center Left Defender,90.0,1.0,0.31,0.0,0.04,2.0,2.0,0.0,104.0,4.0,14.0,0.86,9.0,14.0,5.0,4.0
339,Tomás Soucek,Czechia,Center Right Midfielder,90.0,0.0,0.28,0.0,0.01,1.0,0.0,0.0,43.0,1.0,0.0,0.74,2.0,6.0,1.0,1.0


## Team leaderboards

Aggregate to one row per team per game, then summarise across the tournament.
Team per-90 uses **outfield team-minutes ≈ 90 × 10 + keeper**, but the simplest
and most common convention is *per match*, shown here.

In [8]:
# One row per team per game (sum players), keeping the team's result context.
team_game = (df.groupby(["game_id", "team_name", "opp_team_name"], as_index=False)
               .agg({"team_score": "first", "opp_score": "first",
                     **{c: "sum" for c in STAT_COLS}}))

def team_per_match(stat, n=20):
    g = (team_game.groupby("team_name", as_index=False)
                  .agg(games=("game_id", "nunique"),
                       goals=("team_score", "sum"),
                       total=(stat, "sum")))
    g["per_match"] = g["total"] / g["games"]
    return g.sort_values("per_match", ascending=False).head(n).reset_index(drop=True)

print("Team xG per match")
team_per_match("expectedGoals")

Team xG per match


,team_name,games,goals,total,per_match
0,Germany,1,7,4.298,4.298
1,Switzerland,1,1,3.195,3.195
2,South Korea,1,2,2.309,2.309
3,Ivory Coast,1,1,1.521,1.521
4,Mexico,1,2,1.478,1.478
5,United States,1,4,1.419,1.419
6,Morocco,1,1,1.371,1.371
7,Türkiye,1,0,1.359,1.359
8,Sweden,1,5,1.338,1.338
9,Brazil,1,1,1.304,1.304


In [9]:
# Team attacking summary table (per match), sorted by xG.
TEAM_STATS = ["expectedGoals", "totalShots", "shotsOnTarget",
              "bigChanceCreated", "touchesInOppBox", "finalThirdEntries"]
summary = team_game.groupby("team_name").agg(
    games=("game_id", "nunique"),
    goals_for=("team_score", "sum"),
    goals_against=("opp_score", "sum"),
    **{c: (c, "sum") for c in TEAM_STATS},
)
for c in TEAM_STATS:
    summary[c] = (summary[c] / summary["games"]).round(2)
summary.sort_values("expectedGoals", ascending=False)

,games,goals_for,goals_against,expectedGoals,totalShots,shotsOnTarget,bigChanceCreated,touchesInOppBox,finalThirdEntries
team_name,,,,,,,,,
Germany,1,7,1,4.30,26.0,12.0,5.0,63.0,79.0
Switzerland,1,1,1,3.20,26.0,7.0,5.0,42.0,80.0
South Korea,1,2,1,2.31,15.0,6.0,3.0,24.0,65.0
Ivory Coast,1,1,0,1.52,15.0,4.0,1.0,39.0,34.0
Mexico,1,2,0,1.48,16.0,4.0,2.0,20.0,53.0
United States,1,4,1,1.42,16.0,6.0,3.0,53.0,77.0
Morocco,1,1,1,1.37,14.0,3.0,1.0,13.0,66.0
Türkiye,1,0,2,1.36,30.0,8.0,2.0,51.0,85.0
Sweden,1,5,1,1.34,13.0,7.0,3.0,22.0,47.0


## xG over- / under-performance

Players whose actual goals beat (or trail) their expected goals — finishing
form or luck, depending on your priors.

In [10]:
fin = (df.groupby(["player_name", "team_name"], as_index=False)
         .agg(minutes=("minutes", "sum"),
              goals=("totalGoals", "sum"),
              xG=("expectedGoals", "sum"),
              shots=("totalShots", "sum")))
fin["xG"] = fin["xG"].round(2)
fin["G_minus_xG"] = (fin["goals"] - fin["xG"]).round(2)
fin[fin.shots > 0].sort_values("G_minus_xG", ascending=False).head(15)

,player_name,team_name,minutes,goals,xG,shots,G_minus_xG
364,Yasin Ayari,Sweden,90.0,2.0,0.07,2.0,1.93
116,Folarin Balogun,United States,72.0,2.0,0.47,5.0,1.53
194,Kai Havertz,Germany,90.0,2.0,1.01,4.0,0.99
73,Crysencio Summerville,Netherlands,70.0,1.0,0.02,1.0,0.98
72,Connor Metcalfe,Australia,90.0,1.0,0.03,1.0,0.97
279,Omar Rekik,Tunisia,90.0,1.0,0.03,1.0,0.97
354,Virgil van Dijk,Netherlands,90.0,1.0,0.06,1.0,0.94
353,Vinícius Júnior,Brazil,90.0,1.0,0.08,1.0,0.92
125,Giovanni Reyna,United States,8.0,1.0,0.08,1.0,0.92
197,Keito Nakamura,Japan,90.0,1.0,0.08,3.0,0.92


---
### Where to go next

- Filter by `position` / `position_abbr` for role-specific leaderboards.
- Use `df90` (already has `*_p90` columns) for per-game-level per-90 work.
- Join `glossary` to label any column on the fly with `describe("<col>")`.
- Add derived metrics (e.g. `npxG = expectedGoalsNonPenalty`, shot quality =
  `expectedGoals / totalShots`) as new columns — the raw fields are all here.